<a href="https://colab.research.google.com/github/rohitblpprajapat/100-days-of-code/blob/master/colab_convert_dav2_to_tflite.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Convert Depth Anything V2 to TFLite
Run this notebook in Google Colab to generate the `.tflite` model using `ai-edge-torch`, bypassing Windows OS limitations.

In [1]:
!pip install litert-torch torch torchvision --upgrade
!git clone --depth=1 https://github.com/DepthAnything/Depth-Anything-V2.git
!mkdir -p Depth-Anything-V2/checkpoints


  Using cached torch-2.11.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Using cached torchvision-0.26.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.5 kB)
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
  Using cached torchvision-0.25.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.4 kB)
fatal: destination path 'Depth-Anything-V2' already exists and is not an empty directory.


In [2]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install',
     '--force-reinstall', '--no-cache-dir',
     'tensorflow==2.17.0', 'ai-edge-torch'],
    capture_output=True,
    text=True
)

print("STDOUT:\n", result.stdout[-3000:])  # last 3000 chars
print("STDERR:\n", result.stderr[-3000:])
print("Return code:", result.returncode)

STDOUT:
 ━ 40.8/40.8 kB 185.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 202.4 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of ai-edge-tensorflow to determine which version is compatible with other requirements. This could take a while.

The conflict is caused by:
    tensorflow 2.17.0 depends on protobuf!=4.21.0, !=4.21.1, !=4.21.2, !=4.21.3, !=4.21.4, !=4.21.5, <5.0.0dev and >=3.20.3
    tensorboard 2.17.0 depends on protobuf!=4.24.0, <5.0.0 and >=3.19.6
    ai-edge-tensorflow 2.21.0.dev20251110 depends on protobuf<8.0.0 and >=6.31.1

To fix this you could try to:
1. loosen the range of package versions you've specified
2. remove package versions to allow pip to attempt to solve the dependency conflict


STDERR:
 ERROR: Cannot install litert-torch, tensorflow and tensorflow==2.17.0 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resol

In [3]:
import subprocess, sys

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(f"RC={r.returncode}:", ' '.join(cmd)[:80])
    if r.returncode != 0:
        print("ERR:", r.stderr[-500:])
    return r.returncode

# 1. Uninstall standard tensorflow only
run([sys.executable, '-m', 'pip', 'uninstall', '-y',
     'tensorflow', 'tensorflow-cpu', 'tensorflow-gpu',
     'tensorflow-estimator', 'keras'])

# 2. Nuke leftover .so files that pip uninstall misses
run(['find', '/usr/local/lib/python3.12/dist-packages/tensorflow',
     '-name', '*.so', '-delete'])

# 3. Install litert-torch ALONE — let it resolve all its own deps
run([sys.executable, '-m', 'pip', 'install',
     'litert-torch', '--no-cache-dir'])

print("\n✅ Done — RESTART KERNEL before importing anything")

RC=0: /usr/bin/python3 -m pip uninstall -y tensorflow tensorflow-cpu tensorflow-gpu te
RC=0: find /usr/local/lib/python3.12/dist-packages/tensorflow -name *.so -delete
RC=0: /usr/bin/python3 -m pip install litert-torch --no-cache-dir

✅ Done — RESTART KERNEL before importing anything


In [5]:
import subprocess, sys

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(f"RC={r.returncode}:", ' '.join(cmd)[:80])
    if r.returncode != 0:
        print("ERR:", r.stderr[-800:])

# Wipe everything
run([sys.executable, '-m', 'pip', 'uninstall', '-y',
     'litert-torch', 'ai-edge-tensorflow', 'tensorflow',
     'tensorflow-estimator', 'keras', 'keras-nightly'])

run(['find', '/usr/local/lib/python3.12/dist-packages/tensorflow',
     '-name', '*.so', '-delete'])

# Nightly doesn't depend on ai-edge-tensorflow at all
run([sys.executable, '-m', 'pip', 'install',
     'litert-torch-nightly', '--no-cache-dir'])

print("\n✅ RESTART KERNEL NOW")

RC=0: /usr/bin/python3 -m pip uninstall -y litert-torch ai-edge-tensorflow tensorflow 
RC=0: find /usr/local/lib/python3.12/dist-packages/tensorflow -name *.so -delete
RC=0: /usr/bin/python3 -m pip install litert-torch-nightly --no-cache-dir

✅ RESTART KERNEL NOW


In [2]:
import subprocess, sys

r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'torchao', '--upgrade', '--no-cache-dir'],
    capture_output=True, text=True
)
print(r.stdout[-1000:])
print(r.stderr[-500:])
print("RC:", r.returncode)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 43.4 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


RC: 0


In [1]:
import litert_torch
import torch
print("litert_torch imported successfully")
print("torch:", torch.__version__)

litert_torch imported successfully
torch: 2.9.1+cu128


In [2]:
import sys
from pathlib import Path
sys.path.append('Depth-Anything-V2')

# Patch dynamic xformers allocation in DINOv2 to prevent tracing crashes
attn_path = Path('Depth-Anything-V2/depth_anything_v2/dinov2_layers/attention.py')
text = attn_path.read_text()
if 'XFORMERS_AVAILABLE = True' in text:
    attn_path.write_text(text.replace('XFORMERS_AVAILABLE = True', 'XFORMERS_AVAILABLE = False'))
    print('Patched xformers.')

import torch
from depth_anything_v2.dpt import DepthAnythingV2

model_configs = {'vits': {'encoder': 'vits', 'features': 64, 'out_channels': [48, 96, 192, 384]}}
model = DepthAnythingV2(**model_configs['vits'])
model.load_state_dict(torch.load('Depth-Anything-V2/checkpoints/depth_anything_v2_metric_hypersim_vits.pth', map_location='cpu'))
model.eval()
print('Model loaded successfully.')

Model loaded successfully.


In [3]:
import torch
import litert_torch

dummy_input = torch.zeros(1, 3, 518, 518)
print('Converting via ai-edge-torch...')
edge_model = litert_torch.convert(model, (dummy_input,))
tflite_path = 'depth_anything_v2_metric_indoor_small_s518.tflite'
edge_model.export(tflite_path)
print(f'Successfully saved {tflite_path}')

Converting via ai-edge-torch...


(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default

(00:03) [START] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions

(00:09) [ DONE] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions (+00:05)

...

(00:48) [ DONE] LiteRT-Torch Convert > Run LiteRT Converter Passes (+00:03)

(00:48) [ DONE] LiteRT-Torch Convert (+00:48)

(00:00) [START] Write Model to depth_anything_v2_metric_indoor_small_s518.tflite

(00:00) [ DONE] Write Model to depth_anything_v2_metric_indoor_small_s518.tflite (+00:00)

Successfully saved depth_anything_v2_metric_indoor_small_s518.tflite


In [ ]:
from google.colab import files
files.download('depth_anything_v2_metric_indoor_small_s518.tflite')